# ✈️ Orquestación del ETL OpenSky con Prefect  
## Ejecución del flujo `etl_opensky_flow` y scheduling opcional

Este notebook acompaña al pipeline principal de **ETL de Tráfico Aéreo con OpenSky Network**, ya implementado siguiendo la arquitectura **Bronze → Silver → Gold** y documentado en `01_opensky_etl.ipynb`.

Mientras que el notebook anterior se centra en la **ejecución manual paso a paso** (ingesta, limpieza, enriquecimiento y visualizaciones), aquí el foco está en la **orquestación con Prefect**, utilizando la lógica definida en:

- `src/etl_utils.py` → funciones auxiliares de extracción, transformación y guardado  
- `src/etl_opensky_flow.py` → definición del flujo `etl_opensky_flow` (tasks + flow Prefect)

El flujo automatiza el recorrido completo:

- 📥 **Extracción** del snapshot dinámico desde la API pública de OpenSky  
- 🟤 **Bronze** → normalización básica y persistencia cruda en Delta Lake  
- 🥈 **Silver** → limpieza, tipificación, columnas temporales y particionado por hora  
- 🟡 **Gold** → lectura desde Silver, enriquecimiento con metadatos estáticos y guardado final  

---

## 🎯 Objetivo de este notebook

Este notebook está pensado para:

- ejecutar el flujo **`etl_opensky_flow` de forma manual**, desde Prefect  
- verificar que las tareas de cada capa se encadenan correctamente  
- revisar logs de ejecución y comportamiento general del pipeline  
- documentar una posible **ejecución programada** (cron) sin activarla por defecto

No se redefinen transformaciones ni lógica de negocio:  
simplemente se **importa el flujo ya implementado** y se lo ejecuta en un contexto controlado.

---

## 📘 Estructura de este notebook

1. **Configuración mínima e importación del flujo**
2. **Ejecución manual del pipeline `etl_opensky_flow()`**
3. **(Opcional, documentado) Ejecución programada con `serve` y cron**

Este notebook funciona como interfaz de orquestación y validación, complementando al notebook principal de ETL y preparando el proyecto para futuras integraciones con **Prefect Cloud** o despliegues en **Azure**.

## Uso del flujo definido en `src/etl_opensky_flow.py`

El flujo ETL está implementado en `src/etl_opensky_flow.py` y encapsula el pipeline completo:

- extracción desde OpenSky  
- normalización y guardado en Bronze  
- limpieza, tipificación y particionado en Silver  
- enriquecimiento y persistencia final en Gold  

Desde este notebook se puede:

- **Opción A — correr una ejecución única (demo one-off)** para validar la orquestación  
- **Opción B — servir el flow (opcional)** para mantenerlo activo con ejecución programada


In [13]:
import prefect
print("Prefect versión:", prefect.__version__)

Prefect versión: 2.20.9


### Opción A — Corrida manual del flujo (one-off)

Esta modalidad ejecuta el flujo `etl_opensky_flow` **una sola vez**, ideal para validar
que todas las tareas del pipeline funcionan correctamente:

- extracción del snapshot dinámico  
- limpieza y particionado en Silver  
- enriquecimiento con metadatos estáticos  
- guardado final en Gold  

Se recomienda esta opción para pruebas, validación local y depuración.

### Opción A — Corrida manual del flujo (one-off)

Ejecuta el flujo `etl_opensky_flow` **una sola vez**, ideal para validar que la orquestación funciona correctamente. Permite verificar:

- que la extracción desde OpenSky responde  
- que las transformaciones Bronze → Silver → Gold se encadenan sin errores  
- que el flujo persiste los datos en el Data Lake como se espera  

Esta modalidad es la recomendada para pruebas locales y depuración antes de activar cualquier programación automática.


In [1]:
import sys
import os

# Se agrega la carpeta src al path (sube un nivel desde /notebooks)
sys.path.append(os.path.abspath("../src"))

In [2]:
import importlib

# Importa el módulo de orquestación
etl = importlib.import_module("etl_opensky_flow")

# Ejecuta el flujo ETL de forma local (modo recomendado dentro del Notebook)
etl.etl_opensky_flow()

# Nota:
# También puede ejecutarse desde la terminal:
#     python src/etl_opensky_flow.py
#
# O desde el notebook:
#     !python ../src/etl_opensky_flow.py
#
# Todas las opciones ejecutan exactamente el mismo flow.

17:17:16.636 | INFO    | prefect.engine - Created flow run 'melodic-peacock' for flow 'etl-opensky-full-pipeline'

17:17:16.636 | INFO    | Flow run 'melodic-peacock' - View at https://app.prefect.cloud/account/1513bf29-3686-40b8-9dbf-c85ba6a6f8c0/workspace/377710aa-6343-48a1-a52d-8fd9be6fbba7/flow-runs/flow-run/06928b1c-c453-7460-8000-1f6d1824aec1

17:17:17.386 | INFO    | Flow run 'melodic-peacock' - Created task run 'task_extract_states-0' for task 'task_extract_states'

17:17:17.386 | INFO    | Flow run 'melodic-peacock' - Executing 'task_extract_states-0' immediately...

17:17:20.772 | INFO    | Task run 'extract-opensky-states' - Finished in state Completed()

...

17:17:30.582 | ERROR   | Task run 'load-silver-aircraft-metadata' - Encountered exception during execution:
Traceback (most recent call last):
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\engine.py", line 2169, in orchestrate_task_run
    result = await call.aresult()
             ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\calls.py", line 327, in aresult
    return await asyncio.wrap_future(self.future)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\calls.py", line 352, in _run_sync
    result = self.fn(*self.args, **self.kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\OneDrive\Documentos\Repositorios_notebooks\ETL_OpenSky_Aviation\src\etl_opensky_flow.py", line 167, in task_load_silver_metadata
    df = read_all_from_delta(SILVER_STATIC)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\OneDrive\Documentos\Repositorios_notebooks\ETL_OpenSky_Aviation\src\etl_utils.py", line 213, in read_all_from_delta
    dt = DeltaTable(path)
         ^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\deltalake\table.py", line 166, in __init__
    self._table = RawDeltaTable(
                  ^^^^^^^^^^^^^^
_internal.TableNotFoundError: Local path "data/etl_datalake/silver/api_opensky/aircraft_metadata" does not exist or you don't have access!

17:17:30.816 | ERROR   | Task run 'load-silver-aircraft-metadata' - Finished in state Failed('Task run encountered an exception TableNotFoundError: Local path "data/etl_datalake/silver/api_opensky/aircraft_metadata" does not exist or you don\'t have access!')

17:17:31.064 | ERROR   | Flow run 'melodic-peacock' - Encountered exception during execution:
Traceback (most recent call last):
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\engine.py", line 894, in orchestrate_flow_run
    result = await flow_call.aresult()
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\calls.py", line 327, in aresult
    return await asyncio.wrap_future(self.future)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\calls.py", line 352, in _run_sync
    result = self.fn(*self.args, **self.kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\OneDrive\Documentos\Repositorios_notebooks\ETL_OpenSky_Aviation\src\etl_opensky_flow.py", line 227, in etl_opensky_flow
    df_metadata_loaded = task_load_silver_metadata()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\tasks.py", line 712, in __call__
    return enter_task_run_engine(
           ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\engine.py", line 1469, in enter_task_run_engine
    return from_sync.wait_for_call_in_loop_thread(begin_run)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\api.py", line 218, in wait_for_call_in_loop_thread
    return call.result()
           ^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\calls.py", line 318, in result
    return self.future.result(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\calls.py", line 179, in result
    return self.__get_result()
           ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\calls.py", line 389, in _run_async
    result = await coro
             ^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\engine.py", line 1605, in get_task_call_return_value
    return await future._result()
           ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\futures.py", line 237, in _result
    return await final_state.result(raise_on_failure=raise_on_failure, fetch=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\states.py", line 91, in _get_state_result
    raise await get_state_exception(state)
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\engine.py", line 2169, in orchestrate_task_run
    result = await call.aresult()
             ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\calls.py", line 327, in aresult
    return await asyncio.wrap_future(self.future)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\anaconda3\envs\etl_opensky\Lib\site-packages\prefect\_internal\concurrency\calls.py", line 352, in _run_sync
    result = self.fn(*self.args, **self.kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\OneDrive\Documentos\Repositorios_notebooks\ETL_OpenSky_Aviation\src\etl_opensky_flow.py", line 167, in task_load_silver_metadata
    df = read_all_from_delta(SILVER_STATIC)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\elias\OneDrive\Documentos\Repositorios_notebooks\ETL_OpenSky_Aviation\s

17:17:31.386 | ERROR   | Flow run 'melodic-peacock' - Finished in state Failed('Flow run encountered an exception. TableNotFoundError: Local path "data/etl_datalake/silver/api_opensky/aircraft_metadata" does not exist or you don\'t have access!')

TableNotFoundError: Local path "data/etl_datalake/silver/api_opensky/aircraft_metadata" does not exist or you don't have access!